# Hierarchischer Ansatz: Vergleich der Klassifikatoren

**Autor:** jd · **Datum:** 2026-10-12

**Ziel:** Alle hierarchischen Klassifikatoren vergleichen, gemischte Stufen prüfen und die Fehleranalyse aktualisieren (#104).

In [ ]:
%load_ext autoreload
%autoreload 2

import mlflow
import pandas as pd
from mlflow import MlflowClient
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from awp2.config import (
    CROP_COL,
    LOGISTIC_REGRESSION_MAX_ITER,
    MLFLOW_EXPERIMENT,
    MLFLOW_TRACKING_URI,
    MODEL_DOCS_FIGURES_DIR,
)
from awp2.data import combined_label, load_split
from awp2.evaluation import Metrics, as_target_frame, error_breakdown
from awp2.experiment import RunConfig, TuneConfig, out_of_fold_predictions, run, tune
from awp2.models import (
    BalancedLinearDiscriminantAnalysis,
    HierarchicalClassifier,
    oracle_crop_metrics,
)
from awp2.plots import plot_error_breakdown, plot_metric_comparison, save_doc_figure
from awp2.preprocessing import PreprocessingConfig

In [ ]:
split = load_split()
split.X_val.shape

## 1. Alle hierarchischen Klassifikatoren

Die Bewertungsläufe der Serie (#87, #98–#103) werden über ihre Run-ID aus MLflow geladen –
Metriken, CV-Score des Tuning-Laufs und das gespeicherte Modell. Nichts wird neu eingestellt.
Die Rangfolge gilt nach dem CV-Score; die Validierung bestätigt sie nur.

In [ ]:
EVALUATION_RUNS = {
    "RBF-SVM": "aa082323fa7445e787029399812d08e7",
    "Extra Trees → RBF-SVM": "8449d63ed56d4eb2842591fff940b7d6",
    "Extra Trees": "55dd4258d7ad4f60be3ba9f6543926c1",
    "MLP": "5aa76d10f9c148a6a5c6296dafb45bcf",
    "HistGradientBoosting": "6cc2984fea59453e8b4e6e117f169c66",
    "Logistische Regression": "78fbacd9fda04080900368602427cbbb",
    "kNN": "4be28bd76a9f4e978f154ccf0fd2fb34",
    "PLS-DA": "b21da81ca786446986139a3c1ba8835f",
    "LDA": "2b5eb7dcf4f349bda6df6948fa21d483",
}
client = MlflowClient(MLFLOW_TRACKING_URI)
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
experiment_id = client.get_experiment_by_name(MLFLOW_EXPERIMENT).experiment_id
runs = {label: client.get_run(run_id) for label, run_id in EVALUATION_RUNS.items()}
pipelines = {
    label: mlflow.sklearn.load_model(
        mlflow.search_logged_models(
            experiment_ids=[experiment_id],
            filter_string=f"source_run_id = '{run_id}'",
            output_format="list",
        )[0].model_uri
    )
    for label, run_id in EVALUATION_RUNS.items()
}
predictions = {
    label: as_target_frame(pipeline.predict(split.X_val), split.y_val.index)
    for label, pipeline in pipelines.items()
}

In [ ]:
metric_fields = [name for name in Metrics.model_fields if name != "invalid_combinations"]
ranking = pd.DataFrame(
    {
        label: {
            "CV": client.get_run(r.data.tags["tuning_run"]).data.metrics["cv_bacc_combined_mean"],
            **{name: r.data.metrics[name] for name in ["bacc_crop", "bacc_stage", "bacc_combined"]},
            **error_breakdown(split.y_val, predictions[label]).model_dump(),
            "Oracle-Kultur": oracle_crop_metrics(
                pipelines[label], split.X_val, split.y_val
            ).bacc_combined,
            "Run": r.info.run_id[:8],
        }
        for label, r in runs.items()
    }
).T.sort_values("CV", ascending=False)
ranking

In [ ]:
metrics = {
    label: Metrics(**{name: r.data.metrics[name] for name in metric_fields})
    for label, r in runs.items()
}
ranking_ax = plot_metric_comparison(metrics, metrics=("bacc_combined",))
ranking_ax.set_title("Hierarchischer Ansatz: alle Klassifikatoren (Validierung)")
ranking_ax.legend(fontsize="x-small", ncol=3, loc="lower center")
save_doc_figure(ranking_ax.figure, "hierarchical_vergleich", MODEL_DOCS_FIGURES_DIR)

In [ ]:
breakdown_ax = plot_error_breakdown(
    {label: error_breakdown(split.y_val, predictions[label]) for label in ranking.index}
)
breakdown_ax.set_title("Fehlerzerlegung aller hierarchischen Klassifikatoren")
save_doc_figure(breakdown_ax.figure, "hierarchical_vergleich_zerlegung", MODEL_DOCS_FIGURES_DIR)

## 2. Hilft eine Mischung der Stufen?

Jede Stufe kann einen anderen Klassifikator nutzen. Eine CV-Suche kombiniert die besten
Einstellungen der schnellen Klassifikatoren mit Klassengewichten bzw. gleichen Priors auf beiden
Stufen. Nur wenn eine Mischung die reine SVM in der CV schlägt, wird sie einmal bewertet.

In [ ]:
crop_candidates = [
    SVC(class_weight="balanced", C=30.0, gamma=0.01),
    LogisticRegression(class_weight="balanced", C=10.0, max_iter=LOGISTIC_REGRESSION_MAX_ITER),
    KNeighborsClassifier(n_neighbors=1, metric="cosine", weights="distance"),
]
stage_candidates = [
    SVC(class_weight="balanced", C=100.0, gamma=0.003),
    LogisticRegression(class_weight="balanced", C=100.0, max_iter=LOGISTIC_REGRESSION_MAX_ITER),
    BalancedLinearDiscriminantAnalysis(solver="lsqr", shrinkage=0.01),
    KNeighborsClassifier(n_neighbors=1, metric="cosine", weights="distance"),
]
preprocessing = PreprocessingConfig(use_spectral_standard_scale=True)
mixed = tune(
    HierarchicalClassifier(crop_candidates[0], stage_candidates[0]),
    TuneConfig(
        name="hierarchical_mixed_search",
        approach="hierarchical",
        description="Mischung der Stufen: bestes Kultur- und bestes Stadium-Modell aus #87, "
        "#98, #99, #101 (feste Einstellungen) per CV kombiniert.",
        preprocessing=preprocessing,
        param_grid={"crop_estimator": crop_candidates, "stage_estimator": stage_candidates},
    ),
)
mixed_table = mixed.candidates.assign(
    Kultur=[type(p["crop_estimator"]).__name__ for p in mixed.candidates["params"]],
    Stadium=[type(p["stage_estimator"]).__name__ for p in mixed.candidates["params"]],
)[["Kultur", "Stadium", "mean", "std", "rank"]].sort_values("rank")
mixed_table.round(4)

In [ ]:
best_is_pure_svm = isinstance(mixed.best_params["crop_estimator"], SVC) and isinstance(
    mixed.best_params["stage_estimator"], SVC
)
if best_is_pure_svm:
    mixed_result = {"Entscheidung": "RBF-SVM auf beiden Stufen bleibt; kein neuer Lauf"}
else:
    mixed_result = run(
        mixed.best_model,
        RunConfig(
            name="hierarchical_mixed",
            approach="hierarchical",
            description="Beste Mischung der Stufen aus hierarchical_mixed_search, einmal bewertet.",
            preprocessing=preprocessing,
            tuning_run=mixed.run_id,
        ),
    ).metrics.model_dump()
mixed_result, mixed.run_id

## 3. Ergänzen sich die Klassifikatoren?

Wie viele Zeilen trifft *mindestens ein* Klassifikator? Der Abstand zur SVM ist eine Obergrenze
dessen, was ein Ensemble gewinnen könnte – ein Hinweis, keine erreichbare Zahl. Gerechnet wird
zuerst auf Out-of-fold-Vorhersagen des Trainingsteils (Grundlage für die Wahl von
Ensemble-Kandidaten), die Validierung dient nur als Gegenprobe. Achtung: Extra Trees → RBF-SVM und
RBF-SVM teilen dieselbe Stadium-SVM, die Modelle sind also nicht unabhängig.

In [ ]:
preprocessing_matches = all(
    r.data.params["preprocessing.use_spectral_standard_scale"] == "True" for r in runs.values()
)
oof_predictions = {
    label: out_of_fold_predictions(
        clone(pipelines[label][-1]),
        RunConfig(
            name="hierarchical_oof",
            approach="hierarchical",
            preprocessing=preprocessing,
            balance_samples=runs[label].data.params["balance_samples"] == "True",
            track=False,
        ),
    )
    for label in pipelines
}
preprocessing_matches

In [ ]:
def complementarity(predicted: dict[str, pd.DataFrame], truth: pd.DataFrame) -> pd.Series:
    correct = pd.DataFrame(
        {label: combined_label(pred) == combined_label(truth) for label, pred in predicted.items()}
    )
    crop_correct = pd.DataFrame(
        {label: pred[CROP_COL] == truth[CROP_COL] for label, pred in predicted.items()}
    )
    svm_crop_wrong = ~crop_correct["RBF-SVM"]
    return pd.Series(
        {
            "RBF-SVM richtig": correct["RBF-SVM"].mean(),
            "mindestens ein Klassifikator richtig": correct.any(axis=1).mean(),
            "SVM falsch, mindestens ein anderer richtig": (
                ~correct["RBF-SVM"] & correct.any(axis=1)
            ).mean(),
            "Kultur: SVM falsch (Zeilen)": svm_crop_wrong.sum(),
            "Kultur: SVM falsch, ein anderer richtig (Zeilen)": (
                svm_crop_wrong & crop_correct.drop(columns="RBF-SVM").any(axis=1)
            ).sum(),
        }
    )


pd.DataFrame(
    {
        "Training (out-of-fold)": complementarity(oof_predictions, split.y_train),
        "Validierung": complementarity(predictions, split.y_val),
    }
).round(3)

In [ ]:
oof_crop_correct = pd.DataFrame(
    {label: pred[CROP_COL] == split.y_train[CROP_COL] for label, pred in oof_predictions.items()}
)
svm_misses = ~oof_crop_correct["RBF-SVM"]
pd.Series(
    {
        label: (svm_misses & oof_crop_correct[label]).sum()
        for label in oof_crop_correct.columns.drop("RBF-SVM")
    },
    name="Kulturfehler der SVM, die dieses Modell trifft (Training, out-of-fold)",
).sort_values(ascending=False)

## Befund

- Nach CV ist die RBF-SVM auf beiden Stufen der beste hierarchische Klassifikator (CV 0.839,
  Validierung 0.844); jede Mischung der Stufen ist in der CV schlechter.
- Lineare Modelle verlieren vor allem auf der Kulturstufe, MLP und kNN auch beim Stadium.
- Die Klassifikatoren ergänzen sich – auf dem Trainingsteil (out-of-fold) trifft mindestens einer
  96,2 % der Zeilen (SVM 87,7 %). Die komplementärsten Modelle sind dort kNN, logistische
  Regression und Extra Trees: Kandidaten für einen Ensemble-Versuch per CV.